# 11j — Weekly identifiability, and the spread of the dispersion term

The per-week contact model (`constant_contacts=false`) fits a **hierarchical spatio-temporal GP**: a per-week level `c_t`, a matrix-normal structure field, and a **block-linear × week dispersion** (the *variance term*).

Since 2026-08-02 that dispersion has **no per-cell random effect** — it is simply

$$\log d_{ij,t} \;=\; \beta_{\ell(i,j),t},\qquad \ell(i,j)\in\{\text{c}\to\text{c},\,\text{c}\to\text{a},\,\text{a}\to\text{c},\,\text{a}\to\text{a}\}$$

so all 49 ordered age-pair cells within a child/adult block share one value each week. A per-cell RE lived here briefly: first a flat non-centred hierarchy $\tau_t z_{ij,t}$ (`-hd`, 2026-07-30), then a **regularised horseshoe** $\tau\tilde\lambda_{ij,t}z_{ij,t}$ (`-rhs`, 2026-08-02). Both were removed. Measured across $\tau_0\in\{0.1, 0.01, 0.005, 0.001\}$ at this origin, the horseshoe's global scale was simply outbid by the likelihood ($\tau$ landing 7–15 prior SDs out, the slab inflating until it never bound, $\lambda$ never leaving its init) until $\tau_0=0.001$, where the RE vanished outright — a cliff rather than a usable shrinkage dial. With 49 ordered cells per week, many of them empty, the per-cell dispersion was never identified by the data.

This notebook is **read-only** — it reconstructs everything from the cached `8j_s1_*` Stage-1 chains (no refit, no Stage 2).

**§1 — Moment timelines.** The weekly **mean** degree `⟨k⟩` (`MeanNGM` C0) and **neighbourhood-mean** degree `⟨k²⟩/⟨k⟩·g` (`NeighbourhoodDegreeNGM` C0), with **90% CIs**, for a **contactee** of age **25-34** (bin 4) and **70+** (bin 7), one panel per **contactor** age *i*. The mean depends only on the level; the neighbourhood mean is driven by the second moment, hence by the dispersion term. So **wider ribbons or larger week-to-week jumps in the neighbourhood line than the mean line** are the signature of that variance term — and show which weeks are (un)identifiable.

**§2 — Dispersion spread.** The direct check that the per-cell RE really is gone, against the retained `-hd` chains. Note the two families' dispersion parameters read in **opposite directions**: for `unweighted-negbin` it is the NegBin dispersion **φ**, for `weighted-hweibull` it is the Weibull **shape κ**, where *smaller* κ means a heavier tail, i.e. *more* dispersion.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

default_plot_setting()

In [ ]:
cfg  = FrameworkConfig(constant_contacts = false)   # per-week GP — required for weekly fluctuation
grid = cis_age_grid()
@assert grid.LAB[4] == "25-34" && grid.LAB[7] == "70+"

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)

ORIGIN = Date(2021, 5, 9)
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
# Fail loudly if this origin has not been fitted — otherwise every panel silently comes back blank.
for d in ("unweighted-negbin", "weighted-hweibull")
    f = joinpath("..", "dt_intermediate",
                 "8j_s1_$(d)_$(contacts_label(cfg))_$(ORIGIN)_h1.jld2")
    @assert isfile(f) "no Stage-1 chain for $(d) at $(ORIGIN) — run 8j first ($(f))"
end

win = WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
wd  = load_window_data(win; grid = grid)

println("origin        : ", ORIGIN)
println("contacts token: ", contacts_label(cfg), "   (previous generation: ", CONTACTS_TOKEN_HD, ")")
println("contactees    : ", grid.LAB[4], " (j=4), ", grid.LAB[7], " (j=7)")
println("dispersion    : block-linear × week, no per-cell random effect")

In [ ]:
models = [NegBinAgePair(), HurdleWeibullAgePair()]
include("8j_viz_utils.jl")    # stage1_chain_path
include("10j_viz_utils.jl")   # reconstruct_dispersion_draws, plot_dispersion_cells
include("11j_viz_utils.jl")   # moment_timeline_stats, plot_moment_timeline, plot_within_block_sd
CONTACTEES = (4, 7)           # 25-34, 70+
println("degree models : ", degree_label.(models))

## §1 — Weekly mean vs neighbourhood-mean degree (90% CI)

Four figures = 2 contactees × {`unweighted-negbin`, `weighted-hweibull`}. Each is a 7-panel grid (contactor age *i*); steelblue = mean `⟨k⟩`, darkorange = neighbourhood `⟨k²⟩/⟨k⟩`, ribbons = 90% CI, dashed line = forecast origin. Faint gray bars (secondary right axis) = the per-cell sample size `n_pos = n_roster·(1−p⁰)` (positive contacts in that cell that week) — read the CI width against it: a wide ribbon over thin bars is a sample-starved, poorly-identified week. Saved to `../res/11j_moment_timeline_*`.

In [ ]:
# unweighted-negbin — contactee 25-34 then 70+
for j in CONTACTEES
    display(plot_moment_timeline(NegBinAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end

In [ ]:
# weighted-hweibull — contactee 25-34 then 70+ (K1 = (1−p⁰)·μW; neighbourhood carries the variance term)
for j in CONTACTEES
    display(plot_moment_timeline(HurdleWeibullAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end

## §2 — Is there any within-block structure left?

With `log d_{ij,t} = β_{ℓ(i,j),t}` every ordered cell inside a child/adult block shares one value, so:

1. **Within-block SD of log-dispersion must be identically 0** for the current generation, at every week in every block. The `-hd` line on the same panel shows the spread the old per-cell random effect used to produce. Anything non-zero on the current line means a per-cell term survived the revert — this is the verification, not a decorative comparison. Both cache generations sit on disk under different tokens *and* different directories, so this needs no refit.

2. **The per-cell dispersion map is four flat quadrants**, by construction. That is now the *correct* reading — between 2026-07-30 and 2026-08-02 flat quadrants instead meant the RE scale had collapsed to ~0, so don't carry that interpretation over.

3. The legacy `-hd` chains also record what was given up: `plot_tau_over_weeks` shows how large the per-cell scale τ_t actually was, week by week.

In [ ]:
# (1) VERIFICATION — within-block SD of log dispersion, current model vs the legacy `-hd` hierarchy.
#     The current line must be IDENTICALLY 0 in every block and every week.
for dm in models
    display(plot_within_block_sd(dm, ORIGIN, cfg, grid))
end

In [ ]:
# (2) per-cell dispersion map at the origin week — FOUR FLAT QUADRANTS is the correct result here.
#     unweighted-negbin -> phi ; weighted-hweibull -> Weibull shape kappa (smaller = MORE dispersed)
WK = cfg.smax + cfg.n_fit          # origin week = last window week (the one the NGM is frozen at)
for dm in models
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    display(plot_dispersion_cells(lbl, ORIGIN, cfg, grid;
                                  weighted = is_weighted(dm), week_index = WK))
end

In [ ]:
# (3) what the removed random effect used to be: the legacy `-hd` per-week scale tau_t.
#     Reads dt_intermediate_hierarchical/ (both token AND save_dir differ from the current chains).
for dm in models
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_tau_over_weeks(lbl, ORIGIN, cfg, win.all_weeks; h = 1,
                            contacts = CONTACTS_TOKEN_HD, save_dir = CONTACTS_SAVE_DIR_HD)
    p === nothing || display(p)
end